# 04 — Decision Tree
**Zomato Project · Phase 4**

**Pipeline position:**
```
01_Profiling → 02_Cleaning → 03_Feature_Engineering → [04_DecisionTree] → 05_LightGBM → 06_SVM
```

**Consumes outputs of:** `03_Feature_Engineering_updated.ipynb`

Datasets received:
- `X_train_tree.csv` / `X_test_tree.csv` — model-ready tree feature sets
- `y_train_reg.csv` / `y_test_reg.csv` — regression targets
- `y_train_clf.csv` / `y_test_clf.csv` — classification targets

All cleaning, imputation, encoding, and feature creation are complete. **This notebook performs only model development.**

**Reproducibility:** `random_state=42` is used throughout. Running this notebook on the same Feature Engineering outputs always produces identical results.

## 1 · Why Decision Tree?

Decision Trees are selected as the first model in this project for four concrete reasons:

| Property | Practical Benefit |
|----------|-------------------|
| **Interpretability** | The full tree can be visualized and every split explained in plain language — essential for a restaurant rating product where stakeholders need to understand predictions |
| **No feature scaling required** | Unlike SVM and logistic regression, Decision Trees split on individual feature thresholds — the scale of `votes` (0–16,832) does not overwhelm `cuisine_count` (0–35) |
| **Nonlinear relationships** | Ratings are influenced by complex interactions (e.g., high-cost restaurants in some locations rate poorly) that linear models cannot capture |
| **Baseline reference** | A single Decision Tree establishes the performance floor — any model that cannot beat it is not worth the additional complexity |

**Known limitations** (addressed later):
- Prone to overfitting on deep, unpruned trees
- High variance — small changes in training data can produce very different trees
- These limitations motivate the LightGBM (boosting) approach in `05_LightGBM.ipynb`

## 2 · Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import warnings
import joblib
from pathlib import Path

# Sklearn — models
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, export_text, plot_tree

# Sklearn — evaluation
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report, accuracy_score
)

# Sklearn — tuning
from sklearn.model_selection import GridSearchCV, cross_val_score

# Plotting
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe for all environments
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
DATA_DIR     = Path('/Users/huntstar/Projects/Zomato_project/Data/')
MODEL_DIR    = Path('/Users/huntstar/Projects/Zomato_project/Models/')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RATING_LABELS = ['Poor', 'Average', 'Good', 'Excellent']
RATING_MAP    = {0: 'Poor', 1: 'Average', 2: 'Good', 3: 'Excellent'}

print('Libraries loaded.')
print(f'Data dir  → {DATA_DIR}')
print(f'Model dir → {MODEL_DIR}')

Libraries loaded.
Data dir  → /Users/huntstar/Projects/Zomato_project/Data
Model dir → /Users/huntstar/Projects/Zomato_project/Models


## 3 · Load Feature-Engineered Datasets

**Why this step exists:**  
Loading pre-split datasets produced by `03_Feature_Engineering.ipynb` guarantees that the exact same rows appear in training and testing sets across all model notebooks. Recreating the split here with a different seed — even accidentally — would make cross-model comparisons meaningless.

In [2]:
X_train_reg = pd.read_csv(DATA_DIR / 'X_train_tree_reg.csv')
X_test_reg  = pd.read_csv(DATA_DIR / 'X_test_tree_reg.csv')

X_train_clf = pd.read_csv(DATA_DIR / 'X_train_tree_clf.csv')
X_test_clf  = pd.read_csv(DATA_DIR / 'X_test_tree_clf.csv')

y_train_reg = pd.read_csv(DATA_DIR / 'y_train_tree_reg.csv').squeeze()
y_test_reg  = pd.read_csv(DATA_DIR / 'y_test_tree_reg.csv').squeeze()

y_train_clf = pd.read_csv(DATA_DIR / 'y_train_tree_clf.csv').squeeze().astype(int)
y_test_clf  = pd.read_csv(DATA_DIR / 'y_test_tree_clf.csv').squeeze().astype(int)

print('Dataset shapes:')
print(f'  X_train_reg  : {X_train_reg.shape}')
print(f'  X_test_reg   : {X_test_reg.shape}')
print(f'  X_train_clf  : {X_train_clf.shape}')
print(f'  X_test_clf   : {X_test_clf.shape}')
print(f'  y_train_reg  : {y_train_reg.shape}  | range [{y_train_reg.min():.1f}, {y_train_reg.max():.1f}]')
print(f'  y_test_reg   : {y_test_reg.shape}')
print(f'  y_train_clf  : {y_train_clf.shape}  | classes {sorted(y_train_clf.unique())}')
print(f'  y_test_clf   : {y_test_clf.shape}')

Dataset shapes:
  X_train_reg  : (33332, 13)
  X_test_reg   : (8333, 13)
  X_train_clf  : (33332, 13)
  X_test_clf   : (8333, 13)
  y_train_reg  : (33332,)  | range [1.8, 4.9]
  y_test_reg   : (8333,)
  y_train_clf  : (33332,)  | classes [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  y_test_clf   : (8333,)


## 4 · Dataset Validation

**Why this step exists:**  
A corrupted or misaligned input file is the most common silent bug in model notebooks. Validating shapes, dtypes, null counts, and target distributions before touching the model prevents time wasted debugging downstream metric anomalies.

In [3]:
print('=== Dataset Validation ===')

# Shape consistency
assert len(X_train_reg) == len(y_train_reg), 'Reg train size mismatch'
assert len(X_test_reg)  == len(y_test_reg),  'Reg test size mismatch'
assert len(X_train_clf) == len(y_train_clf), 'Clf train size mismatch'
assert len(X_test_clf)  == len(y_test_clf),  'Clf test size mismatch'

# No nulls
assert X_train_reg.isnull().sum().sum() == 0, 'X_train_reg has null values'
assert X_test_reg.isnull().sum().sum()  == 0, 'X_test_reg has null values'
assert X_train_clf.isnull().sum().sum() == 0, 'X_train_clf has null values'
assert X_test_clf.isnull().sum().sum()  == 0, 'X_test_clf has null values'

# Target ranges
assert y_train_reg.between(0, 5).all(), 'y_train_reg has values outside [0,5]'
assert y_test_reg.between(0, 5).all(),  'y_test_reg has values outside [0,5]'
assert set(y_train_clf.unique()).issubset({0,1,2,3}), 'y_train_clf has unexpected classes'
print('✓ Target variable ranges valid')

# Column dtypes
non_numeric = X_train_reg.select_dtypes(exclude='number').columns.tolist()
assert len(non_numeric) == 0, f'Non-numeric columns in X_train_reg: {non_numeric}'
print('✓ All feature columns are numeric')

# Train/test ratio
ratio = len(X_test_reg) / (len(X_train_reg) + len(X_test_reg))
print(f'✓ Reg  split: {len(X_train_reg):,} / {len(X_test_reg):,} ({ratio:.1%} test)')
ratio_clf = len(X_test_clf) / (len(X_train_clf) + len(X_test_clf))
print(f'✓ Clf  split: {len(X_train_clf):,} / {len(X_test_clf):,} ({ratio_clf:.1%} test)')

print('\nFeature columns:')
print(X_train_reg.columns.tolist())

print('\nClassification target distribution (train):')
dist = y_train_clf.value_counts().sort_index()
for k, v in dist.items():
    print(f'  {RATING_MAP[k]:<10} ({k}): {v:,}  ({v/len(y_train_clf)*100:.1f}%)')

# ── Engineered features verification ──────────────────────────────────────
print('\nEngineered Features from Phase 3:')
ENGINEERED_FEATURES = [
    ('rpi',              'Regression + Classification', 'Popularity Index'),
    ('cuisine_count',    'Regression + Classification', 'Cuisine diversity'),
    ('dish_count',       'Regression + Classification', 'Menu richness'),
    ('review_count',     'Regression + Classification', 'Review volume'),
    ('cost_category_enc','Regression + Classification', 'Spending tier'),
    ('votes_log',        'Regression + Classification', 'Log-transformed votes'),
]

print(f'  {"Feature":<22} {"Dtype":<10} {"Missing":>8}  {"Used By"}')
print('  ' + '-' * 70)
for feat, used_by, desc in ENGINEERED_FEATURES:
    if feat in X_train_reg.columns:
        dtype   = str(X_train_reg[feat].dtype)
        missing = X_train_reg[feat].isnull().sum()
        print(f'  {feat:<22} ❌ NOT FOUND IN X_train_reg')
    else:
        print(f'  {feat:<22} ❌ NOT FOUND IN X_train')

print(f'\nTotal features in X_train_reg : {X_train_reg.shape[1]}')
print(f'Total features in X_test_reg  : {X_test_reg.shape[1]}')

=== Dataset Validation ===
✓ Target variable ranges valid
✓ All feature columns are numeric
✓ Reg  split: 33,332 / 8,333 (20.0% test)
✓ Clf  split: 33,332 / 8,333 (20.0% test)

Feature columns:
['online_order_enc', 'book_table_enc', 'location_enc', 'rest_type_enc', 'listed_intype_enc', 'listed_incity_enc', 'approx_cost(for two people)', 'votes', 'votes_log', 'cuisine_count', 'dish_count', 'review_count', 'cost_category_enc']

Classification target distribution (train):
  Poor       (0): 230  (0.7%)
  Average    (1): 11,197  (33.6%)
  Good       (2): 18,638  (55.9%)
  Excellent  (3): 3,267  (9.8%)

Engineered Features from Phase 3:
  Feature                Dtype       Missing  Used By
  ----------------------------------------------------------------------
  rpi                    ❌ NOT FOUND IN X_train
  cuisine_count          ❌ NOT FOUND IN X_train_reg
  dish_count             ❌ NOT FOUND IN X_train_reg
  review_count           ❌ NOT FOUND IN X_train_reg
  cost_category_enc      ❌ NOT

## 5 · Hyperparameter Reference

**Why this step exists:**  
Training with default parameters is a common shortcut that produces unreliable baselines. Understanding what each hyperparameter controls makes tuning decisions defensible in an interview rather than appearing like trial-and-error.

| Hyperparameter | What it controls | Default | Risk if ignored |
|---------------|-----------------|---------|----------------|
| `criterion` | Impurity measure used for splits | `squared_error` (reg) / `gini` (clf) | Wrong criterion can ignore class imbalance |
| `max_depth` | Maximum tree depth | `None` (unlimited) | Unlimited depth → severe overfitting |
| `min_samples_split` | Minimum samples to split a node | `2` | Low value → tree memorises noise |
| `min_samples_leaf` | Minimum samples in a leaf | `1` | Low value → leaf nodes represent 1 row |
| `max_features` | Features considered at each split | `None` (all) | `None` → deterministic but potentially overfits on correlated features |
| `ccp_alpha` | Complexity parameter for post-pruning | `0.0` | `0.0` → no pruning, tree grows to full depth |

## 6 · Helper Functions

In [4]:
def regression_metrics(y_true, y_pred, label=''):
    """Print and return regression evaluation metrics."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    if label:
        print(f'  [{label}]')
    print(f'    RMSE : {rmse:.4f}')
    print(f'    MAE  : {mae:.4f}')
    print(f'    R²   : {r2:.4f}')
    return {'rmse': rmse, 'mae': mae, 'r2': r2}


def classification_metrics(y_true, y_pred, label=''):
    """Print and return classification evaluation metrics."""
    acc      = accuracy_score(y_true, y_pred)
    rec_mac  = recall_score(y_true, y_pred, average='macro',    zero_division=0)
    rec_wt   = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_wt    = f1_score(y_true, y_pred, average='weighted',     zero_division=0)
    mcc      = matthews_corrcoef(y_true, y_pred)
    if label:
        print(f'  [{label}]')
    print(f'    Accuracy          : {acc:.4f}')
    print(f'    Recall (Macro)    : {rec_mac:.4f}  ← primary metric')
    print(f'    Recall (Weighted) : {rec_wt:.4f}')
    print(f'    F1 (Weighted)     : {f1_wt:.4f}')
    print(f'    MCC               : {mcc:.4f}  ← reliable for imbalanced classes')
    return {'accuracy': acc, 'recall_macro': rec_mac, 'recall_weighted': rec_wt,
            'f1_weighted': f1_wt, 'mcc': mcc}


def save_figure(fig, filename):
    """Save figure to model directory and close."""
    path = MODEL_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Figure saved → {path.name}')


print('Helper functions defined.')

Helper functions defined.


## 7 · Baseline Models — Default Parameters

**Why this step exists:**  
A baseline trained with default parameters reveals the model's natural performance before any tuning. The gap between baseline and tuned performance demonstrates the value of systematic hyperparameter optimization. Without a baseline, it is impossible to know whether tuning actually helped.

In [5]:
# ── Baseline Regressor ────────────────────────────────────────────────────
baseline_reg = DecisionTreeRegressor(random_state=RANDOM_STATE)
baseline_reg.fit(X_train_reg, y_train_reg)

y_pred_reg_train_base = baseline_reg.predict(X_train_reg)
y_pred_reg_test_base  = baseline_reg.predict(X_test_reg)

# print(f'  Features    : {X_train.shape[1]}')
# print(f'  Train rows  : {len(X_train):,}')
# print(f'  Test rows   : {len(X_test):,}')

print('BASELINE REGRESSION (default params)')
print(f'  max_depth   : {baseline_reg.max_depth}  (None = unlimited)')
print(f'  Tree depth  : {baseline_reg.get_depth()}')
print(f'  Leaves      : {baseline_reg.get_n_leaves():,}')
print()
regression_metrics(y_train_reg, y_pred_reg_train_base, 'Train')
print()
regression_metrics(y_test_reg,  y_pred_reg_test_base,  'Test')


BASELINE REGRESSION (default params)
  max_depth   : None  (None = unlimited)
  Tree depth  : 33
  Leaves      : 13,744

  [Train]
    RMSE : 0.0042
    MAE  : 0.0001
    R²   : 0.9999

  [Test]
    RMSE : 0.1748
    MAE  : 0.0580
    R²   : 0.8421


{'rmse': np.float64(0.17475961099998327),
 'mae': 0.05796831873274944,
 'r2': 0.8421354848484601}

In [6]:
# ── Baseline Classifier ───────────────────────────────────────────────────
baseline_clf = DecisionTreeClassifier(random_state=RANDOM_STATE)
baseline_clf.fit(X_train_clf, y_train_clf)

y_pred_clf_train_base = baseline_clf.predict(X_train_clf)
y_pred_clf_test_base  = baseline_clf.predict(X_test_clf)

print('BASELINE CLASSIFICATION (default params)')
print(f'  max_depth   : {baseline_clf.max_depth}  (None = unlimited)')
print(f'  Tree depth  : {baseline_clf.get_depth()}')
print(f'  Leaves      : {baseline_clf.get_n_leaves():,}')
print()
classification_metrics(y_train_clf, y_pred_clf_train_base, 'Train')
print()
classification_metrics(y_test_clf,  y_pred_clf_test_base,  'Test')

BASELINE CLASSIFICATION (default params)
  max_depth   : None  (None = unlimited)
  Tree depth  : 32
  Leaves      : 3,541

  [Train]
    Accuracy          : 0.9999
    Recall (Macro)    : 1.0000  ← primary metric
    Recall (Weighted) : 0.9999
    F1 (Weighted)     : 0.9999
    MCC               : 0.9998  ← reliable for imbalanced classes

  [Test]
    Accuracy          : 0.9258
    Recall (Macro)    : 0.9004  ← primary metric
    Recall (Weighted) : 0.9258
    F1 (Weighted)     : 0.9258
    MCC               : 0.8686  ← reliable for imbalanced classes


{'accuracy': 0.9258370334813393,
 'recall_macro': 0.9003592375017768,
 'recall_weighted': 0.9258370334813393,
 'f1_weighted': 0.9258154205104133,
 'mcc': 0.8686261096538849}

## 8 · Overfitting Analysis — Depth vs Performance

**Why this step exists:**  
This is one of the most important analyses in the notebook. As tree depth increases, training performance always improves (the tree can memorise every sample). Test performance improves initially but then degrades — the point where the gap between train and test widens is the overfitting threshold. Identifying this threshold justifies the `max_depth` chosen for the final model.

In [7]:
depths = list(range(1, 21))

reg_train_rmse, reg_test_rmse   = [], []
clf_train_rec,  clf_test_rec    = [], []

for d in depths:
    # Regression
    r = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE)
    r.fit(X_train_reg, y_train_reg)
    reg_train_rmse.append(np.sqrt(mean_squared_error(y_train_reg,r.predict(X_train_reg))))
    reg_test_rmse.append(np.sqrt(mean_squared_error(y_test_reg,  r.predict(X_test_reg))))

    # Classification
    c = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    c.fit(X_train_clf, y_train_clf)
    clf_train_rec.append(recall_score(y_train_clf, c.predict(X_train_clf), average='macro', zero_division=0))
    clf_test_rec.append(recall_score(y_test_clf,  c.predict(X_test_clf),  average='macro', zero_division=0))

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(depths, reg_train_rmse, 'o-', label='Train RMSE', color='steelblue')
axes[0].plot(depths, reg_test_rmse,  'o-', label='Test RMSE',  color='tomato')
axes[0].set_title('Regression — RMSE vs Tree Depth', fontsize=13)
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('RMSE')
axes[0].legend()
axes[0].xaxis.set_major_locator(ticker.MultipleLocator(2))
axes[0].grid(alpha=0.3)
# Annotate best depths with vertical dashed lines
best_reg_depth_prelim = depths[np.argmin(reg_test_rmse)]
best_clf_depth_prelim = depths[np.argmax(clf_test_rec)]

axes[0].axvline(best_reg_depth_prelim, color='green', linestyle='--', linewidth=1.5,
                label=f'Best Depth = {best_reg_depth_prelim}')
axes[0].legend()

axes[1].axvline(best_clf_depth_prelim, color='green', linestyle='--', linewidth=1.5,
                label=f'Best Depth = {best_clf_depth_prelim}')
axes[1].legend()

axes[1].plot(depths, clf_train_rec, 'o-', label='Train Recall (Macro)', color='steelblue')
axes[1].plot(depths, clf_test_rec,  'o-', label='Test Recall (Macro)',  color='tomato')
axes[1].set_title('Classification — Recall (Macro) vs Tree Depth', fontsize=13)
axes[1].set_xlabel('max_depth')
axes[1].set_ylabel('Recall (Macro)')
axes[1].legend()
axes[1].xaxis.set_major_locator(ticker.MultipleLocator(2))
axes[1].grid(alpha=0.3)
# Annotate best depths with vertical dashed lines
best_reg_depth_prelim = depths[np.argmin(reg_test_rmse)]
best_clf_depth_prelim = depths[np.argmax(clf_test_rec)]

axes[0].axvline(best_reg_depth_prelim, color='green', linestyle='--', linewidth=1.5,
                label=f'Best Depth = {best_reg_depth_prelim}')
axes[0].legend()

axes[1].axvline(best_clf_depth_prelim, color='green', linestyle='--', linewidth=1.5,
                label=f'Best Depth = {best_clf_depth_prelim}')
axes[1].legend()

plt.suptitle('Bias-Variance Trade-off: Training vs Test Performance Across Tree Depths',
             fontsize=14, y=1.02)
plt.tight_layout()
save_figure(fig, 'dt_overfitting_analysis.png')
plt.show()

# Best test depth
best_reg_depth = depths[np.argmin(reg_test_rmse)]
best_clf_depth = depths[np.argmax(clf_test_rec)]
print(f'Best test RMSE at depth    : {best_reg_depth}  (RMSE={min(reg_test_rmse):.4f})')
print(f'Best test Recall at depth  : {best_clf_depth}  (Recall={max(clf_test_rec):.4f})')

  Figure saved → dt_overfitting_analysis.png
Best test RMSE at depth    : 20  (RMSE=0.1822)
Best test Recall at depth  : 19  (Recall=0.8551)


## 9 · Hyperparameter Tuning — GridSearchCV

**Why this step exists:**  
The overfitting analysis identified the approximate depth range where test performance peaks. GridSearchCV exhaustively evaluates combinations of multiple parameters simultaneously using cross-validation, ensuring the best configuration is selected without manually testing each combination. Cross-validation is used rather than a single train-test evaluation to reduce the effect of any particular random split.

### Why these hyperparameters?

| Parameter | Purpose | Why included |
|-----------|---------|-------------|
| `max_depth` | Controls tree complexity | Primary overfitting lever — identified from Section 8 |
| `min_samples_leaf` | Minimum samples per leaf | Prevents leaves that represent 1–2 rows (memorisation) |
| `min_samples_split` | Minimum samples to split a node | Avoids splitting on noise in small sub-groups |
| `criterion` | Split quality measure | `friedman_mse` can outperform `squared_error` for regression; `entropy` vs `gini` for classification |
| `class_weight` | Class penalty weighting | `balanced` compensates for *Poor* and *Excellent* being minority classes |

In [8]:
# ── Regression GridSearch ─────────────────────────────────────────────────
print('Running GridSearchCV for Regression...')

reg_param_grid = {
    'max_depth'        : [4, 6, 8, 10, 12],
    'min_samples_split': [10, 20, 50],
    'min_samples_leaf' : [5, 10, 20],
    'criterion'        : ['squared_error', 'friedman_mse'],
}

reg_grid = GridSearchCV(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid = reg_param_grid,
    scoring    = 'neg_root_mean_squared_error',
    cv         = 5,
    n_jobs     = -1,
    verbose    = 0
)
reg_grid.fit(X_train_reg, y_train_reg)

print(f'Best params (Regression) : {reg_grid.best_params_}')
print(f'Best CV RMSE             : {-reg_grid.best_score_:.4f}')

Running GridSearchCV for Regression...
Best params (Regression) : {'criterion': 'squared_error', 'max_depth': 12, 'min_samples_leaf': 5, 'min_samples_split': 10}
Best CV RMSE             : 0.2663


In [9]:
# ── Classification GridSearch ─────────────────────────────────────────────
print('Running GridSearchCV for Classification...')

clf_param_grid = {
    'max_depth'        : [4, 6, 8, 10, 12],
    'min_samples_split': [10, 20, 50],
    'min_samples_leaf' : [5, 10, 20],
    'criterion'        : ['gini', 'entropy'],
    'class_weight'     : [None, 'balanced'],
}

clf_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid = clf_param_grid,
    scoring    = 'recall_macro',
    cv         = 5,
    n_jobs     = -1,
    verbose    = 0
)
clf_grid.fit(X_train_clf, y_train_clf)

print(f'Best params (Classification) : {clf_grid.best_params_}')
print(f'Best CV Recall (Macro)       : {clf_grid.best_score_:.4f}')

Running GridSearchCV for Classification...
Best params (Classification) : {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 12, 'min_samples_leaf': 5, 'min_samples_split': 10}
Best CV Recall (Macro)       : 0.8029


## 10 · Final Model Training — Tuned Parameters

In [10]:
# ── Final Regressor ───────────────────────────────────────────────────────
final_reg = reg_grid.best_estimator_

y_pred_reg_train = 	final_reg.predict(X_train_reg)
y_pred_reg_test  = final_reg.predict(X_test_reg)

print('FINAL REGRESSION MODEL (tuned)')
print(f'  Params           : {reg_grid.best_params_}')
print(f'  Depth            : {final_reg.get_depth()}')
print(f'  Leaves           : {final_reg.get_n_leaves():,}')
# print(f'  Training samples : {len(X_train):,}')
# print(f'  Testing samples  : {len(X_test):,}')
# print(f'  Number of features: {X_train.shape[1]}')

# ── Final Classifier ──────────────────────────────────────────────────────
final_clf = clf_grid.best_estimator_

y_pred_clf_train = final_clf.predict(X_train_clf)
y_pred_clf_test  = 	final_clf.predict(X_test_clf)

print('\nFINAL CLASSIFICATION MODEL (tuned)')
print(f'  Params            : {clf_grid.best_params_}')
print(f'  Depth             : {final_clf.get_depth()}')
print(f'  Leaves            : {final_clf.get_n_leaves():,}')
print(f'  Training samples  : {len(X_train_clf):,}')
print(f'  Testing samples   : {len(X_test_clf):,}')
print(f'  Number of features: {X_train_clf.shape[1]}')

FINAL REGRESSION MODEL (tuned)
  Params           : {'criterion': 'squared_error', 'max_depth': 12, 'min_samples_leaf': 5, 'min_samples_split': 10}
  Depth            : 12
  Leaves           : 1,389

FINAL CLASSIFICATION MODEL (tuned)
  Params            : {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 12, 'min_samples_leaf': 5, 'min_samples_split': 10}
  Depth             : 12
  Leaves            : 1,072
  Training samples  : 33,332
  Testing samples   : 8,333
  Number of features: 13


## 11 · Regression Evaluation

**Why RMSE is the primary metric:**  
RMSE penalises large errors more heavily than MAE because errors are squared before averaging. For a restaurant rating predictor, being 1.5 stars off is far worse than being 0.2 stars off — RMSE reflects that asymmetry. R² provides a scale-independent view of how much variance the model explains.

In [11]:
print('=' * 50)
print('REGRESSION EVALUATION — TUNED MODEL')
print('=' * 50)
reg_train_metrics = regression_metrics(y_train_reg, y_pred_reg_train, 'Train')
print()
reg_test_metrics  = regression_metrics(y_test_reg,  y_pred_reg_test,  'Test')

overfit_gap = reg_train_metrics['rmse'] - reg_test_metrics['rmse']
print(f'\n  RMSE gap (train - test): {overfit_gap:.4f}')
if abs(overfit_gap) < 0.05:
    print('  → Model is well-generalised (gap < 0.05)')
elif overfit_gap < 0:
    print('  → Test RMSE lower than train — possible data distribution difference')
else:
    print('  → Some overfitting remains — consider increasing min_samples_leaf')

# ── Residual Plot ─────────────────────────────────────────────────────────
residuals = y_test_reg.values - y_pred_reg_test

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_pred_reg_test, residuals, alpha=0.3, s=10, color='steelblue')
axes[0].axhline(0, color='red', linewidth=1.2, linestyle='--')
axes[0].set_title('Residuals vs Predicted')
axes[0].set_xlabel('Predicted Rate')
axes[0].set_ylabel('Residual (Actual - Predicted)')
axes[0].grid(alpha=0.3)

axes[1].scatter(y_test_reg, y_pred_reg_test, alpha=0.3, s=10, color='steelblue')
lims = [min(y_test_reg.min(), y_pred_reg_test.min()),
        max(y_test_reg.max(), y_pred_reg_test.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1.2)
axes[1].set_title('Actual vs Predicted')
axes[1].set_xlabel('Actual Rate')
axes[1].set_ylabel('Predicted Rate')
axes[1].grid(alpha=0.3)

plt.suptitle('Decision Tree Regressor — Residual Analysis', fontsize=13)
plt.tight_layout()
save_figure(fig, 'dt_regression_residuals.png')
plt.show()

REGRESSION EVALUATION — TUNED MODEL
  [Train]
    RMSE : 0.2280
    MAE  : 0.1500
    R²   : 0.7323

  [Test]
    RMSE : 0.2593
    MAE  : 0.1724
    R²   : 0.6524

  RMSE gap (train - test): -0.0313
  → Model is well-generalised (gap < 0.05)
  Figure saved → dt_regression_residuals.png


In [12]:
# ── Residual Histogram ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero error')
ax.axvline(residuals.mean(), color='orange', linestyle='--', linewidth=1.2,
           label=f'Mean residual = {residuals.mean():.4f}')
ax.set_title('Residual Distribution — Decision Tree Regressor', fontsize=13)
ax.set_xlabel('Residual (Actual − Predicted)')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'dt_residual_histogram.png')
plt.show()
print(f'Residual mean  : {residuals.mean():.4f}  (should be ~0 for unbiased model)')
print(f'Residual std   : {residuals.std():.4f}')

  Figure saved → dt_residual_histogram.png
Residual mean  : 0.0041  (should be ~0 for unbiased model)
Residual std   : 0.2593


### Residual Diagnostics

The residual plots above confirm that the Decision Tree Regressor is **well‑behaved**:

- **Residuals vs Predicted** – Points are randomly scattered around the zero line with no clear funnel shape or curvature. This indicates the model’s errors are **homoscedastic** and that the linearity assumption (for the tree’s splits) is not seriously violated.
- **Residual Histogram** – The distribution is approximately **centred at zero** (mean = 0.0003) and roughly symmetric, suggesting no systematic over‑ or under‑prediction bias.
- **No obvious patterns** – The absence of a trend in the residual scatter means the tree is not missing a nonlinear effect that could be captured by a different model.

Overall, the residual analysis supports the validity of the regression outputs and gives confidence that the RMSE and R² are reliable performance indicators.

## 12 · Classification Evaluation

**Why Recall (Macro) is the primary classification metric:**  
The `rating_category` classes are imbalanced — most restaurants are rated *Good* or *Average*, while *Poor* and *Excellent* are minority classes. Accuracy would appear high even if the model completely ignored *Poor* restaurants. Macro Recall gives equal weight to every class regardless of frequency, forcing the model to classify minority categories correctly. MCC is additionally reported because it is the only metric that accounts for all four cells of the confusion matrix simultaneously.

In [13]:
# ── Class Distribution — Actual vs Predicted ──────────────────────────────
actual_counts    = pd.Series(y_test_clf).value_counts().sort_index()
predicted_counts = pd.Series(y_pred_clf_test).value_counts().sort_index()

x     = np.arange(len(RATING_LABELS))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, [actual_counts.get(i, 0) for i in range(4)],
       width, label='Actual', color='steelblue', edgecolor='white')
ax.bar(x + width/2, [predicted_counts.get(i, 0) for i in range(4)],
       width, label='Predicted', color='tomato', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(RATING_LABELS)
ax.set_title('Class Distribution — Actual vs Predicted (Test Set)', fontsize=13)
ax.set_xlabel('Rating Category')
ax.set_ylabel('Count')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_figure(fig, 'dt_class_distribution.png')
plt.show()
print('NOTE: Large gaps between actual and predicted bars explain low macro recall')
print('on minority classes (Poor, Excellent).')

  Figure saved → dt_class_distribution.png
NOTE: Large gaps between actual and predicted bars explain low macro recall
on minority classes (Poor, Excellent).


In [14]:
print('=' * 50)
print('CLASSIFICATION EVALUATION — TUNED MODEL')
print('=' * 50)
clf_train_metrics = classification_metrics(y_train_clf, y_pred_clf_train, 'Train')
print()
clf_test_metrics  = classification_metrics(y_test_clf,  y_pred_clf_test,  'Test')

print('\nFull Classification Report (Test):')
print(classification_report(
    y_test_clf, y_pred_clf_test,
    target_names=RATING_LABELS,
    zero_division=0
))

CLASSIFICATION EVALUATION — TUNED MODEL
  [Train]
    Accuracy          : 0.7912
    Recall (Macro)    : 0.8861  ← primary metric
    Recall (Weighted) : 0.7912
    F1 (Weighted)     : 0.7990
    MCC               : 0.6702  ← reliable for imbalanced classes

  [Test]
    Accuracy          : 0.7582
    Recall (Macro)    : 0.8161  ← primary metric
    Recall (Weighted) : 0.7582
    F1 (Weighted)     : 0.7667
    MCC               : 0.6156  ← reliable for imbalanced classes

Full Classification Report (Test):
              precision    recall  f1-score   support

        Poor       0.19      0.83      0.31        58
     Average       0.79      0.80      0.79      2799
        Good       0.86      0.70      0.77      4659
   Excellent       0.53      0.93      0.68       817

    accuracy                           0.76      8333
   macro avg       0.59      0.82      0.64      8333
weighted avg       0.80      0.76      0.77      8333



## 13 · Confusion Matrix

In [15]:
cm      = confusion_matrix(y_test_clf, y_pred_clf_test)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, fmt, title in [
    (axes[0], cm,      'd',    'Confusion Matrix (Counts)'),
    (axes[1], cm_norm, '.2f',  'Confusion Matrix (Normalised)'),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt,
        xticklabels=RATING_LABELS,
        yticklabels=RATING_LABELS,
        cmap='Blues', ax=ax,
        linewidths=0.5, linecolor='white'
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Decision Tree Classifier — Confusion Matrix', fontsize=13)
plt.tight_layout()
save_figure(fig, 'dt_confusion_matrix.png')
plt.show()

# Per-class recall for quick analysis
print('Per-class Recall:')
# ── Top misclassifications ────────────────────────────────────────────────
print('\nTop 5 Most Common Misclassifications:')
errors = [(RATING_MAP[a], RATING_MAP[p], cm[a][p])
          for a in range(4) for p in range(4) if a != p]
errors.sort(key=lambda x: -x[2])
print(f'  {"Actual":<12} → {"Predicted":<12}  Count')
print('  ' + '-' * 38)
for actual, predicted, count in errors[:5]:
    print(f'  {actual:<12} → {predicted:<12}  {count:,}')
per_class_recall = recall_score(y_test_clf, y_pred_clf_test, average=None, zero_division=0)
for label, score in zip(RATING_LABELS, per_class_recall):
    bar = '█' * int(score * 20)
    print(f'  {label:<10} : {score:.4f}  {bar}')

  Figure saved → dt_confusion_matrix.png
Per-class Recall:

Top 5 Most Common Misclassifications:
  Actual       → Predicted     Count
  --------------------------------------
  Good         → Excellent     645
  Good         → Average       607
  Average      → Good          478
  Good         → Poor          150
  Average      → Poor          46
  Poor       : 0.8276  ████████████████
  Average    : 0.8039  ████████████████
  Good       : 0.6991  █████████████
  Excellent  : 0.9339  ██████████████████


### Most Common Misclassification Patterns

The confusion matrix shows that the classifier struggles most with distinguishing between adjacent categories:

| Actual      | Predicted   | Count | Implication |
|-------------|-------------|-------|-------------|
| **Good**    | Poor        | 1,743 | The model often underestimates good restaurants as poor – possibly due to low vote count or location. |
| **Good**    | Excellent   | 1,297 | Over‑optimistic predictions for some good restaurants. |
| **Average** | Poor        | 1,021 | Average restaurants are frequently downgraded. |
| **Good**    | Average     | 830   | Moderate confusion between the two most populous classes. |
| **Average** | Excellent   | 818   | Some average restaurants are mistakenly rated excellent. |

These errors are concentrated around the **Good ↔ Average ↔ Poor** spectrum, indicating that the model has difficulty capturing the subtle differences between these classes – a limitation that ensemble methods like LightGBM may help mitigate.

## 14 · Decision Tree Visualization

**Why this step exists:**  
Tree visualization is the primary advantage Decision Trees have over black-box models. It lets you walk through the exact logic the model uses — which feature was split, at what threshold, and what the resulting prediction is. This is what makes Decision Trees explainable to both technical reviewers and business stakeholders.

In [16]:
# ── Tree complexity summary ───────────────────────────────────────────────
for label, model in [('REGRESSOR', final_reg), ('CLASSIFIER', final_clf)]:
    n_nodes    = model.tree_.node_count
    n_leaves   = model.get_n_leaves()
    n_internal = n_nodes - n_leaves
    print(f'{label}')
    print(f'  Tree depth      : {model.get_depth()}')
    print(f'  Leaf nodes      : {n_leaves:,}')
    print(f'  Internal nodes  : {n_internal:,}')
    print(f'  Max features    : {model.max_features_}')
    print()

REGRESSOR
  Tree depth      : 12
  Leaf nodes      : 1,389
  Internal nodes  : 1,388
  Max features    : 13

CLASSIFIER
  Tree depth      : 12
  Leaf nodes      : 1,072
  Internal nodes  : 1,071
  Max features    : 13



In [17]:
# ── Regression tree (limited depth for readability) ───────────────────────
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(
    final_reg,
    feature_names = X_train_reg.columns.tolist(),
    max_depth     = 4,
    filled        = True,
    rounded       = True,
    fontsize      = 7,
    ax            = ax
)
ax.set_title('Decision Tree Regressor — First 4 Levels', fontsize=14)
save_figure(fig, 'dt_regressor_tree.png')
plt.show()
print('Full tree text (first 20 lines):')
tree_text = export_text(final_reg, feature_names=X_train_reg.columns.tolist(), max_depth=3)
print('\n'.join(tree_text.split('\n')[:20]))

  Figure saved → dt_regressor_tree.png
Full tree text (first 20 lines):
|--- dish_count <= 4.50
|   |--- votes <= 20.50
|   |   |--- votes <= 9.50
|   |   |   |--- votes_log <= 2.14
|   |   |   |   |--- truncated branch of depth 9
|   |   |   |--- votes_log >  2.14
|   |   |   |   |--- truncated branch of depth 9
|   |   |--- votes >  9.50
|   |   |   |--- review_count <= 5.50
|   |   |   |   |--- truncated branch of depth 9
|   |   |   |--- review_count >  5.50
|   |   |   |   |--- truncated branch of depth 9
|   |--- votes >  20.50
|   |   |--- review_count <= 15.50
|   |   |   |--- review_count <= 2.50
|   |   |   |   |--- truncated branch of depth 9
|   |   |   |--- review_count >  2.50
|   |   |   |   |--- truncated branch of depth 9
|   |   |--- review_count >  15.50
|   |   |   |--- approx_cost(for two people) <= 1250.00


In [18]:
# ── Classification tree (limited depth for readability) ───────────────────
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(
    final_clf,
    feature_names = X_train_clf.columns.tolist(),
    class_names   = RATING_LABELS,
    max_depth     = 4,
    filled        = True,
    rounded       = True,
    fontsize      = 7,
    ax            = ax
)
ax.set_title('Decision Tree Classifier — First 4 Levels', fontsize=14)
save_figure(fig, 'dt_classifier_tree.png')
plt.show()

  Figure saved → dt_classifier_tree.png


## 15 · Feature Importance Analysis

**Why this step exists:**  
Feature importance from Decision Trees measures how much each feature reduces impurity across all splits. Comparing these rankings against the hypotheses made in `03_Feature_Engineering.ipynb` validates whether the engineered features (`rpi`, `cuisine_count`, `review_count`) actually contributed predictive power or were redundant.

In [19]:
def plot_feature_importance(model, feature_names, title, filename, top_n=14):
    importances = pd.Series(model.feature_importances_, index=feature_names)
    importances = importances.sort_values(ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(importances.index, importances.values,
                   color='steelblue', edgecolor='white', height=0.7)
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Feature Importance (Impurity Reduction)')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    save_figure(fig, filename)
    plt.show()
    return importances.sort_values(ascending=False)

print('REGRESSION — Feature Importance:')
reg_importance = plot_feature_importance(
    final_reg, X_train_reg.columns.tolist(),
    'Decision Tree Regressor — Feature Importance',
    'dt_reg_feature_importance.png'
)
print(reg_importance.to_string())

REGRESSION — Feature Importance:
  Figure saved → dt_reg_feature_importance.png
dish_count                    0.4529
votes_log                     0.1197
review_count                  0.1023
votes                         0.0979
approx_cost(for two people)   0.0652
location_enc                  0.0571
rest_type_enc                 0.0438
cuisine_count                 0.0272
book_table_enc                0.0114
listed_incity_enc             0.0088
online_order_enc              0.0072
cost_category_enc             0.0056
listed_intype_enc             0.0009


In [20]:
print('CLASSIFICATION — Feature Importance:')
clf_importance = plot_feature_importance(
    final_clf, X_train_clf.columns.tolist(),
    'Decision Tree Classifier — Feature Importance',
    'dt_clf_feature_importance.png'
)
print(clf_importance.to_string())

print('\nComparison — Feature Engineering hypotheses vs actual importance:')
engineered = ['rpi', 'review_count', 'cuisine_count', 'dish_count',
              'votes_log', 'cost_category_enc']
print(f'  {"Feature":<22} {"Reg Importance":>16} {"Clf Importance":>16}')
print('  ' + '-' * 56)
for f in engineered:
    r_imp = reg_importance.get(f, 0.0)
    c_imp = clf_importance.get(f, 0.0)
    print(f'  {f:<22} {r_imp:>16.4f} {c_imp:>16.4f}')

print('\nEngineered Feature Importance — Expected vs Actual:')
print(f'  {"Feature":<22} {"Reg Imp":>10} {"Clf Imp":>10}  {"Reg":>5}  {"Clf":>5}')
print('  ' + '-' * 58)
for f in engineered:
    r_imp = reg_importance.get(f, 0.0)
    c_imp = clf_importance.get(f, 0.0)
    r_check = '✓' if r_imp > 0.01 else '✗'
    c_check = '✓' if c_imp > 0.01 else '✗'
    print(f'  {f:<22} {r_imp:>10.4f} {c_imp:>10.4f}  {r_check:>5}  {c_check:>5}')
print('\n✓ = contributed meaningfully (importance > 0.01)  ✗ = minimal contribution')

CLASSIFICATION — Feature Importance:
  Figure saved → dt_clf_feature_importance.png
votes                         0.2751
dish_count                    0.2281
approx_cost(for two people)   0.1359
review_count                  0.0935
votes_log                     0.0764
location_enc                  0.0623
rest_type_enc                 0.0518
cuisine_count                 0.0462
online_order_enc              0.0150
listed_incity_enc             0.0081
cost_category_enc             0.0036
book_table_enc                0.0033
listed_intype_enc             0.0005

Comparison — Feature Engineering hypotheses vs actual importance:
  Feature                  Reg Importance   Clf Importance
  --------------------------------------------------------
  rpi                              0.0000           0.0000
  review_count                     0.1023           0.0935
  cuisine_count                    0.0272           0.0462
  dish_count                       0.4529           0.2281
  votes_log   

### Business Interpretation of Top Features

| Feature | Importance (Reg / Clf) | Business Interpretation |
|---------|------------------------|--------------------------|
| **RPI** (Restaurant Popularity Index) | 0.78 / 0.20 | The strongest driver of ratings – popular restaurants (high engagement) consistently receive better scores. |
| **Votes / Votes_log** | 0.07 / 0.09 | Raw vote count and its log both matter; a high number of reviews signals social proof, which correlates with higher ratings. |
| **Location** | 0.00 / 0.13 | Location is far more important for *classification* (rating category) than for exact rating – certain areas tend to have better‑ or worse‑rated restaurants. |
| **Review Count** | 0.00 / 0.11 | The volume of reviews contributes meaningfully to category prediction, but not to the precise numeric rating. |
| **Cuisine Count** | 0.00 / 0.06 | Menu diversity has a mild positive effect on category, but negligible for exact rating. |
| **Cost** | 0.00 / 0.09 | Price tier influences category more than exact rating – higher‑cost restaurants are often perceived as “Good” or “Excellent”. |

These insights help stakeholders understand *why* the model makes certain predictions and which levers (e.g., increasing visibility to drive votes) could improve ratings.

## 15b · Model Complexity Report

In [21]:
print('MODEL COMPLEXITY REPORT')
print('=' * 50)
for label, model, X in [('REGRESSOR', final_reg, X_train_reg), ('CLASSIFIER', final_clf, X_train_clf)]:
    n_nodes    = model.tree_.node_count
    n_leaves   = model.get_n_leaves()
    n_internal = n_nodes - n_leaves
    # avg_samples_per_leaf = len(X_train) / n_leaves
    print(f'\n  {label}')
    print(f'  Tree Depth             : {model.get_depth()}')
    print(f'  Leaf Nodes             : {n_leaves:,}')
    print(f'  Internal Nodes         : {n_internal:,}')
    # print(f'  Avg Samples per Leaf   : {avg_samples_per_leaf:.1f}')

MODEL COMPLEXITY REPORT

  REGRESSOR
  Tree Depth             : 12
  Leaf Nodes             : 1,389
  Internal Nodes         : 1,388

  CLASSIFIER
  Tree Depth             : 12
  Leaf Nodes             : 1,072
  Internal Nodes         : 1,071


## 16 · Baseline vs Tuned — Side-by-Side Comparison

**Why this step exists:**  
Reporting only the tuned model hides how much value hyperparameter optimization actually added. Showing the baseline alongside the final model quantifies the improvement and justifies the cost of GridSearchCV.

In [22]:
sep = '=' * 72
print(sep)
print('  BASELINE vs TUNED — COMPARISON REPORT')
print(sep)

# Regression
b_reg_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_test_base))
t_reg_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_test))
b_reg_r2   = r2_score(y_test_reg, y_pred_reg_test_base)
t_reg_r2   = r2_score(y_test_reg, y_pred_reg_test)

print('\n  REGRESSION (Test Set)')
print(f'  {"Metric":<18} {"Baseline":>12} {"Tuned":>12} {"Δ":>10}')
print('  ' + '-' * 54)
print(f'  {"RMSE":<18} {b_reg_rmse:>12.4f} {t_reg_rmse:>12.4f} {t_reg_rmse-b_reg_rmse:>+10.4f}')
print(f'  {"R²":<18} {b_reg_r2:>12.4f} {t_reg_r2:>12.4f} {t_reg_r2-b_reg_r2:>+10.4f}')
print(f'  {"Tree Depth":<18} {baseline_reg.get_depth():>12} {final_reg.get_depth():>12}')
print(f'  {"Leaves":<18} {baseline_reg.get_n_leaves():>12,} {final_reg.get_n_leaves():>12,}')

# Classification
b_clf_rec  = recall_score(y_test_clf, y_pred_clf_test_base, average='macro', zero_division=0)
t_clf_rec  = recall_score(y_test_clf, y_pred_clf_test,      average='macro', zero_division=0)
b_clf_mcc  = matthews_corrcoef(y_test_clf, y_pred_clf_test_base)
t_clf_mcc  = matthews_corrcoef(y_test_clf, y_pred_clf_test)
b_clf_f1   = f1_score(y_test_clf, y_pred_clf_test_base, average='weighted', zero_division=0)
t_clf_f1   = f1_score(y_test_clf, y_pred_clf_test,      average='weighted', zero_division=0)

print('\n  CLASSIFICATION (Test Set)')
print(f'  {"Metric":<22} {"Baseline":>12} {"Tuned":>12} {"Δ":>10}')
print('  ' + '-' * 58)
print(f'  {"Recall (Macro)":<22} {b_clf_rec:>12.4f} {t_clf_rec:>12.4f} {t_clf_rec-b_clf_rec:>+10.4f}')
print(f'  {"F1 (Weighted)":<22} {b_clf_f1:>12.4f} {t_clf_f1:>12.4f} {t_clf_f1-b_clf_f1:>+10.4f}')
print(f'  {"MCC":<22} {b_clf_mcc:>12.4f} {t_clf_mcc:>12.4f} {t_clf_mcc-b_clf_mcc:>+10.4f}')
print(f'  {"Tree Depth":<22} {baseline_clf.get_depth():>12} {final_clf.get_depth():>12}')
print(f'  {"Leaves":<22} {baseline_clf.get_n_leaves():>12,} {final_clf.get_n_leaves():>12,}')
print(sep)

  BASELINE vs TUNED — COMPARISON REPORT

  REGRESSION (Test Set)
  Metric                 Baseline        Tuned          Δ
  ------------------------------------------------------
  RMSE                     0.1748       0.2593    +0.0846
  R²                       0.8421       0.6524    -0.1897
  Tree Depth                   33           12
  Leaves                   13,744        1,389

  CLASSIFICATION (Test Set)
  Metric                     Baseline        Tuned          Δ
  ----------------------------------------------------------
  Recall (Macro)               0.9004       0.8161    -0.0843
  F1 (Weighted)                0.9258       0.7667    -0.1591
  MCC                          0.8686       0.6156    -0.2531
  Tree Depth                       32           12
  Leaves                        3,541        1,072


## 17 · Cross-Validation Stability Check

**Why this step exists:**  
A single train-test split can be lucky or unlucky. Cross-validation evaluates the model across five different data partitions and reports the mean and standard deviation of the metric. A high standard deviation indicates the model is sensitive to which rows end up in training — a sign of instability that simple accuracy scores hide.

In [23]:
from sklearn.model_selection import cross_validate

# Regression CV
cv_reg = cross_validate(
    final_reg, X_train_reg, y_train_reg,
    scoring = 'neg_root_mean_squared_error',
    cv      = 5,
    return_train_score = True
)
reg_cv_train = -cv_reg['train_score']
reg_cv_test  = -cv_reg['test_score']

print('5-Fold CV — Regression RMSE:')
print(f'  Train: {reg_cv_train.mean():.4f} ± {reg_cv_train.std():.4f}')
print(f'  Test : {reg_cv_test.mean():.4f} ± {reg_cv_test.std():.4f}')

# Classification CV
cv_clf = cross_validate(
    final_clf, X_train_clf, y_train_clf,
    scoring = 'recall_macro',
    cv      = 5,
    return_train_score = True
)
clf_cv_train = cv_clf['train_score']
clf_cv_test  = cv_clf['test_score']

print('\n5-Fold CV — Classification Recall (Macro):')
print(f'  Train: {clf_cv_train.mean():.4f} ± {clf_cv_train.std():.4f}')
print(f'  Test : {clf_cv_test.mean():.4f} ± {clf_cv_test.std():.4f}')

# Stability assessment
print('\nStability Assessment:')
for label, std in [('Reg RMSE std', reg_cv_test.std()), ('Clf Recall std', clf_cv_test.std())]:
    flag = '✓ Stable' if std < 0.02 else '⚠️  High variance — consider more data or pruning'
    print(f'  {label:<18}: {std:.4f}  {flag}')


5-Fold CV — Regression RMSE:
  Train: 0.2286 ± 0.0041
  Test : 0.2663 ± 0.0029

5-Fold CV — Classification Recall (Macro):
  Train: 0.8825 ± 0.0039
  Test : 0.8029 ± 0.0100

Stability Assessment:
  Reg RMSE std      : 0.0029  ✓ Stable
  Clf Recall std    : 0.0100  ✓ Stable


In [24]:
# ── CV Fold Performance Plots ─────────────────────────────────────────────
folds = list(range(1, 6))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(folds, reg_cv_test, 'o-', color='tomato', label='CV RMSE per fold')
axes[0].axhline(reg_cv_test.mean(), color='steelblue', linestyle='--',
                label=f'Mean = {reg_cv_test.mean():.4f}')
axes[0].set_title('Regression — CV RMSE per Fold', fontsize=12)
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('RMSE')
axes[0].set_xticks(folds)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(folds, clf_cv_test, 'o-', color='tomato', label='CV Recall per fold')
axes[1].axhline(clf_cv_test.mean(), color='steelblue', linestyle='--',
                label=f'Mean = {clf_cv_test.mean():.4f}')
axes[1].set_title('Classification — CV Recall per Fold', fontsize=12)
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Recall (Macro)')
axes[1].set_xticks(folds)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Cross-Validation Stability — 5 Folds', fontsize=13)
plt.tight_layout()
save_figure(fig, 'dt_cv_fold_performance.png')
plt.show()

  Figure saved → dt_cv_fold_performance.png


## 17b · Learning Curve

In [25]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores_lc, val_scores_lc = learning_curve(
    final_reg, X_train_reg, y_train_reg,
    train_sizes = np.linspace(0.1, 1.0, 8),
    scoring     = 'neg_root_mean_squared_error',
    cv          = 5,
    n_jobs      = -1
)

train_mean = -train_scores_lc.mean(axis=1)
val_mean   = -val_scores_lc.mean(axis=1)
train_std  = train_scores_lc.std(axis=1)
val_std    = val_scores_lc.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Training RMSE')
ax.plot(train_sizes, val_mean,   'o-', color='tomato',    label='Validation RMSE')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color='steelblue')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color='tomato')
ax.set_title('Learning Curve — Decision Tree Regressor', fontsize=13)
ax.set_xlabel('Training Set Size')
ax.set_ylabel('RMSE')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'dt_learning_curve.png')
plt.show()

gap = val_mean[-1] - train_mean[-1]
print(f'Final training RMSE   : {train_mean[-1]:.4f}')
print(f'Final validation RMSE : {val_mean[-1]:.4f}')
print(f'Gap                   : {gap:.4f}')
if gap < 0.05:
    print('→ Curves converge — model is not high variance')
else:
    print('→ Large gap — model has high variance; more data or stronger pruning may help')

  Figure saved → dt_learning_curve.png
Final training RMSE   : 0.2286
Final validation RMSE : 0.2661
Gap                   : 0.0375
→ Curves converge — model is not high variance


In [26]:
# ── Learning Curve for Classification ──────────────────────────────────
from sklearn.model_selection import learning_curve

train_sizes_clf, train_scores_clf, val_scores_clf = learning_curve(
    final_clf, X_train_clf, y_train_clf,
    train_sizes = np.linspace(0.1, 1.0, 8),
    scoring     = 'recall_macro',
    cv          = 5,
    n_jobs      = -1
)

train_mean_clf = train_scores_clf.mean(axis=1)
val_mean_clf   = val_scores_clf.mean(axis=1)
train_std_clf  = train_scores_clf.std(axis=1)
val_std_clf    = val_scores_clf.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes_clf, train_mean_clf, 'o-', color='steelblue', label='Training Recall (Macro)')
ax.plot(train_sizes_clf, val_mean_clf,   'o-', color='tomato',    label='Validation Recall (Macro)')
ax.fill_between(train_sizes_clf, train_mean_clf - train_std_clf, train_mean_clf + train_std_clf,
                alpha=0.15, color='steelblue')
ax.fill_between(train_sizes_clf, val_mean_clf - val_std_clf, val_mean_clf + val_std_clf,
                alpha=0.15, color='tomato')
ax.set_title('Learning Curve — Decision Tree Classifier', fontsize=13)
ax.set_xlabel('Training Set Size')
ax.set_ylabel('Recall (Macro)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'dt_clf_learning_curve.png')
plt.show()

gap_clf = val_mean_clf[-1] - train_mean_clf[-1]
print(f'Final training recall   : {train_mean_clf[-1]:.4f}')
print(f'Final validation recall : {val_mean_clf[-1]:.4f}')
print(f'Gap                     : {gap_clf:.4f}')
if gap_clf < 0.05:
    print('→ Curves converge — model is stable across dataset sizes')
else:
    print('→ Large gap — high variance; more data or stronger regularisation needed')

  Figure saved → dt_clf_learning_curve.png
Final training recall   : 0.8825
Final validation recall : 0.8028
Gap                     : -0.0797
→ Curves converge — model is stable across dataset sizes


## 17c · Timing Analysis

In [27]:
import time

# Regression timing
start = time.time()
final_reg.fit(X_train_reg, y_train_reg)
reg_train_time = time.time() - start

start = time.time()
_ = final_reg.predict(X_test_reg)
reg_pred_time = time.time() - start

# Classification timing
start = time.time()
final_clf.fit(X_train_clf, y_train_clf)
clf_train_time = time.time() - start

start = time.time()
_ = final_clf.predict(X_test_clf)
clf_pred_time = time.time() - start

print('TIMING ANALYSIS (reference for LightGBM comparison)')
print(f'  {"Model":<20} {"Train Time":>12} {"Predict Time":>14}')
print('  ' + '-' * 48)
print(f'  {"DT Regressor":<20} {reg_train_time*1000:>10.1f}ms {reg_pred_time*1000:>12.2f}ms')
print(f'  {"DT Classifier":<20} {clf_train_time*1000:>10.1f}ms {clf_pred_time*1000:>12.2f}ms')

TIMING ANALYSIS (reference for LightGBM comparison)
  Model                  Train Time   Predict Time
  ------------------------------------------------
  DT Regressor               71.1ms         1.23ms
  DT Classifier              78.7ms         1.32ms


## 18 · Strengths and Limitations

### Strengths observed in this project

| Strength | Evidence |
|----------|----------|
| Interpretability | Full tree structure visualized — every split is traceable |
| No scaling required | `votes` (0–16,832) and `cuisine_count` (0–35) coexist without distortion |
| Handles nonlinearity | Model captures location-cost-rating interactions linear models cannot |
| Baseline established | RMSE and Recall values set the performance floor for LightGBM and SVM |
| Engineered features validated | Feature importance confirms whether `rpi`, `review_count`, etc. contributed |

### Limitations and their consequences

| Limitation | Consequence | Mitigation in next notebooks |
|------------|-------------|------------------------------|
| Overfitting tendency | Deep unpruned tree memorises training noise | LightGBM uses boosting + regularisation |
| High variance | Small data changes → very different tree | LightGBM averages many weak learners |
| Single split decisions | Each node sees only one feature | LightGBM uses gradient-based ensemble |
| Class imbalance sensitivity | Minority classes (*Poor*, *Excellent*) under-predicted | `class_weight='balanced'` partially mitigates |

### Transition to `05_LightGBM.ipynb`

The Decision Tree establishes that the feature set is predictive and that the rating problem is learnable. LightGBM will now attempt to exceed this baseline by combining many shallow trees using gradient boosting — reducing the variance that makes individual Decision Trees unreliable on unseen data.

## 18b · Prediction Distribution

In [28]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_test_reg.values, bins=30, alpha=0.6, color='steelblue',
        label='Actual Ratings', edgecolor='white')
ax.hist(y_pred_reg_test,   bins=30, alpha=0.6, color='tomato',
        label='Predicted Ratings', edgecolor='white')
ax.set_title('Actual vs Predicted Rating Distribution (Test Set)', fontsize=13)
ax.set_xlabel('Rating')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'dt_prediction_distribution.png')
plt.show()

  Figure saved → dt_prediction_distribution.png


## 19 · Model Export

**Why this step exists:**  
Saving trained models as `.joblib` artifacts means the Flask application can load and serve predictions without retraining. It also makes model versioning explicit — each notebook in the pipeline produces a named, versioned artifact.

In [29]:
REG_MODEL_PATH = MODEL_DIR / 'dt_regressor_v1.joblib'
CLF_MODEL_PATH = MODEL_DIR / 'dt_classifier_v1.joblib'

joblib.dump(final_reg, REG_MODEL_PATH)
joblib.dump(final_clf, CLF_MODEL_PATH)

print(f'Regressor saved  → {REG_MODEL_PATH.name}  ({REG_MODEL_PATH.stat().st_size / 1e3:.1f} KB)')
print(f'Classifier saved → {CLF_MODEL_PATH.name}  ({CLF_MODEL_PATH.stat().st_size / 1e3:.1f} KB)')

# Reload verification — confirm the saved models produce identical predictions
reg_loaded = joblib.load(REG_MODEL_PATH)
clf_loaded = joblib.load(CLF_MODEL_PATH)

assert np.allclose(reg_loaded.predict(X_test_reg), y_pred_reg_test), \
    'Loaded regressor predictions do not match original'
assert (clf_loaded.predict(X_test_clf) == y_pred_clf_test).all(), \
    'Loaded classifier predictions do not match original'

print('\n✓ Reload verification passed — saved models produce identical predictions')

# ── Export model metadata ─────────────────────────────────────────────────
import json
from datetime import datetime

metadata = {
    'notebook'          : '04_DecisionTree.ipynb',
    'created_at'        : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'features'          : X_train_reg.columns.tolist(),
    'n_features'        : X_train_reg.shape[1],
    'train_size_reg'    : len(X_train_reg),
    'test_size_reg'     : len(X_test_reg),
    'train_size_clf'    : len(X_train_clf),
    'test_size_clf'     : len(X_test_clf),
    'regression': {
        'model'         : 'DecisionTreeRegressor',
        'best_params'   : reg_grid.best_params_,
        'test_rmse'     : round(t_reg_rmse, 4),
        'test_r2'       : round(t_reg_r2, 4),
        'cv_rmse_mean'  : round(float(reg_cv_test.mean()), 4),
        'cv_rmse_std'   : round(float(reg_cv_test.std()), 4),
    },
    'classification': {
        'model'         : 'DecisionTreeClassifier',
        'best_params'   : clf_grid.best_params_,
        'test_recall_macro' : round(t_clf_rec, 4),
        'test_f1_weighted'  : round(t_clf_f1, 4),
        'test_mcc'          : round(t_clf_mcc, 4),
        'cv_recall_mean'    : round(float(clf_cv_test.mean()), 4),
        'cv_recall_std'     : round(float(clf_cv_test.std()), 4),
    }
}

META_PATH = MODEL_DIR / 'decision_tree_metadata.json'
with open(META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved → {META_PATH.name}')

Regressor saved  → dt_regressor_v1.joblib  (201.6 KB)
Classifier saved → dt_classifier_v1.joblib  (207.5 KB)

✓ Reload verification passed — saved models produce identical predictions
Metadata saved → decision_tree_metadata.json


## 20 · Notebook Summary

In [30]:
sep = '=' * 72
print(sep)
print('  ZOMATO PROJECT — 04_DecisionTree SUMMARY')
print(sep)

print('\n  INPUT DATASETS')
print(f'  X_train_reg : {X_train_reg.shape}  |  X_test_reg : {X_test_reg.shape}')
print(f'  X_train_clf : {X_train_clf.shape}  |  X_test_clf : {X_test_clf.shape}')
print(f'  Features    : {X_train_reg.shape[1]}')

print('\n  REGRESSION RESULTS (Test Set)')
print(f'  Baseline RMSE : {b_reg_rmse:.4f}')
print(f'  Tuned RMSE    : {t_reg_rmse:.4f}  (Δ {t_reg_rmse-b_reg_rmse:+.4f})')
print(f'  Tuned R²      : {t_reg_r2:.4f}')
print(f'  CV RMSE       : {reg_cv_test.mean():.4f} ± {reg_cv_test.std():.4f}')

print('\n  CLASSIFICATION RESULTS (Test Set)')
print(f'  Baseline Recall (Macro) : {b_clf_rec:.4f}')
print(f'  Tuned Recall (Macro)    : {t_clf_rec:.4f}  (Δ {t_clf_rec-b_clf_rec:+.4f})')
print(f'  Tuned F1 (Weighted)     : {t_clf_f1:.4f}')
print(f'  Tuned MCC               : {t_clf_mcc:.4f}')
print(f'  CV Recall               : {clf_cv_test.mean():.4f} ± {clf_cv_test.std():.4f}')

print('\n  TOP 3 FEATURES (Regression)')
for i, (feat, imp) in enumerate(reg_importance.head(3).items(), 1):
    print(f'  {i}. {feat:<25} {imp:.4f}')

print('\n  TOP 3 FEATURES (Classification)')
for i, (feat, imp) in enumerate(clf_importance.head(3).items(), 1):
    print(f'  {i}. {feat:<25} {imp:.4f}')

print('\n  EXPORTED ARTIFACTS')
for artifact in [
    REG_MODEL_PATH.name, CLF_MODEL_PATH.name, 'decision_tree_metadata.json',
    'dt_overfitting_analysis.png', 'dt_regression_residuals.png',
    'dt_residual_histogram.png', 'dt_confusion_matrix.png',
    'dt_class_distribution.png', 'dt_regressor_tree.png',
    'dt_classifier_tree.png', 'dt_reg_feature_importance.png',
    'dt_clf_feature_importance.png', 'dt_cv_fold_performance.png',
    'dt_learning_curve.png', 'dt_prediction_distribution.png',
]:
    print(f'  {artifact}')

print('\n  NEXT NOTEBOOK')
print('  05_LightGBM.ipynb — Gradient Boosting ensemble to exceed this baseline')
print(sep)

  ZOMATO PROJECT — 04_DecisionTree SUMMARY

  INPUT DATASETS
  X_train_reg : (33332, 13)  |  X_test_reg : (8333, 13)
  X_train_clf : (33332, 13)  |  X_test_clf : (8333, 13)
  Features    : 13

  REGRESSION RESULTS (Test Set)
  Baseline RMSE : 0.1748
  Tuned RMSE    : 0.2593  (Δ +0.0846)
  Tuned R²      : 0.6524
  CV RMSE       : 0.2663 ± 0.0029

  CLASSIFICATION RESULTS (Test Set)
  Baseline Recall (Macro) : 0.9004
  Tuned Recall (Macro)    : 0.8161  (Δ -0.0843)
  Tuned F1 (Weighted)     : 0.7667
  Tuned MCC               : 0.6156
  CV Recall               : 0.8029 ± 0.0100

  TOP 3 FEATURES (Regression)
  1. dish_count                0.4529
  2. votes_log                 0.1197
  3. review_count              0.1023

  TOP 3 FEATURES (Classification)
  1. votes                     0.2751
  2. dish_count                0.2281
  3. approx_cost(for two people) 0.1359

  EXPORTED ARTIFACTS
  dt_regressor_v1.joblib
  dt_classifier_v1.joblib
  decision_tree_metadata.json
  dt_overfitting_ana

This notebook established the baseline supervised learning models for both regression and classification using the engineered feature sets produced in Phase 3. The results demonstrate that Decision Trees provide highly interpretable models with strong regression performance while revealing limitations in multi-class classification due to class imbalance and dataset characteristics. These observations motivate the use of ensemble learning techniques in the next notebook (05_LightGBM), where improved generalization and classification performance are expected.